In [ ]:
!pip install  langchain langchain-core langchain-groq

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
llm = ChatGroq(groq_api_key="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",model="llama-3.1-8b-instant")

In [ ]:
llm.invoke("hello").content

'Hello, how are you today?'

In [ ]:
from langchain.tools import tool
from email.mime.text import MIMEText
import smtplib

In [ ]:
@tool
def send_mail(receiver : str, subject : str, body : str)->str:
  """this tool is this send mail to receiver with parameter receiver and subject and body """
  sender = "here_gmail_id"
  password = "here_password"

  msg = MIMEText(body)
  msg["Subject"] = subject
  msg["From"] = sender
  msg["To"] = receiver

  server = smtplib.SMTP("smtp.gmail.com",587)
  server.starttls()
  server.login(sender,password)
  server.send_message(msg)
  server.quit()
  return "Mail sent successfully"


In [ ]:
!pip install langchain-community

In [ ]:
!pip install arxiv

In [ ]:

from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper

In [ ]:
@tool
def tool_arxiv_search(query: str) -> str:
    """Search scientific papers on arXiv"""

    import time

    for _ in range(3):
        try:
            wrapper = ArxivAPIWrapper(
                top_k_results=2,
                doc_content_chars_max=1500
            )
            tool = ArxivQueryRun(api_wrapper=wrapper)
            return tool.run(query)

        except Exception:
            time.sleep(10)

    return "arXiv busy or rate-limited. Try later."

In [ ]:
llm_with_tools = llm.bind_tools([send_mail,tool_arxiv_search])

# Ask mode

In [ ]:
from langchain.agents import create_agent


my_agent = create_agent(llm_with_tools,[send_mail,tool_arxiv_search])

In [ ]:
my_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": """
Use research tool first to find vjepa paper.

Then:
1. Summarize the paper in beginner-friendly language.
2. Include title, authors, year, contributions, use cases.
3. Use send_mail tool after summary is ready.

Receiver: nostalgic1235@gmail.com
Subject: vjepa Research

Follow strict order:
research tool -> summary -> send_mail
"""
        }
    ]
})

{'messages': [HumanMessage(content='\nUse research tool first to find vjepa paper.\n\nThen:\n1. Summarize the paper in beginner-friendly language.\n2. Include title, authors, year, contributions, use cases.\n3. Use send_mail tool after summary is ready.\n\nReceiver: nostalgic1235@gmail.com\nSubject: vjepa Research\n\nFollow strict order:\nresearch tool -> summary -> send_mail\n', additional_kwargs={}, response_metadata={}, id='c41bdf84-2fa2-48ab-99fa-f940e56d56c2'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '851wf7a3b', 'function': {'arguments': '{"query":"vjepa"}', 'name': 'tool_arxiv_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 356, 'prompt_tokens': 382, 'total_tokens': 738, 'completion_time': 0.513895899, 'completion_tokens_details': None, 'prompt_time': 0.025600265, 'prompt_tokens_details': None, 'queue_time': 0.135807775, 'total_time': 0.539496164}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e0